# SLURM
The [Simple Linux Utility for Resource Management (SLURM)](https://slurm.schedmd.com) job scheduler is currently used on most HPC systems, including LANL's flagship machines such as Chicoma and Venado. *Side note: the company behind SLURM named [SchedMD was recently acquired by Nvidia](https://blogs.nvidia.com/blog/nvidia-acquires-schedmd/), but day-to-day usage is unchanged for end users.*

**Why does this matter?**
On a shared HPC cluster you cannot simply `ssh` directly into a compute node and run a simulation interactively. The cluster is a shared resource used simultaneously by many research groups. SLURM acts as a traffic controller: it accepts resource requests, queues jobs, enforces fair-share scheduling, and dispatches work to nodes when the requested resources become available. Understanding SLURM's three core commands is the minimum required to run jobs on any LANL system.

**Learning objectives:** After this notebook you will know how to:
- Submit batch jobs with `sbatch` and inspect their status with `squeue`
- Launch parallel tasks inside an existing allocation with `srun`
- Write and run MPI-parallel Python scripts with `mpi4py`
- Choose between splitting an allocation via multiple `srun` calls vs. splitting a single MPI communicator inside Python — and understand the tradeoffs of each approach

---

## `sbatch`
Submit a batch job script from the **head node** (also called the *login node*) of the HPC system to the job scheduler. The job is queued and dispatched to compute nodes when the requested resources are available; `sbatch` returns immediately after submission so your terminal session is not blocked.

Typical usage:
```bash
sbatch job.sh
```

The script usually contains:

* resource requests (`--nodes`, `--ntasks`, `--time`, etc.)
* environment / module loading
* application launch commands

A list of all available options is available as part of the [SLURM documentation](https://slurm.schedmd.com/sbatch.html).

Example `job.sh` script:
```bash
#!/bin/bash
#SBATCH --output=time.out
#SBATCH --error=error.out
#SBATCH --job-name=test
#SBATCH --chdir=/u/janj/slurmtest
#SBATCH --get-user-env=L
#SBATCH --partition=s.cmfe
#SBATCH --time=60
#SBATCH --ntasks=8

srun hostname
```

This example is going to return the `hostname` eight times, given the number of tasks is set to eight. The `#SBATCH` lines are directives parsed by the scheduler before the script runs; they are comments as far as bash is concerned, so the script is also valid shell.

> **Tip:** Always set `--output` and `--error` so that standard output and standard error from your job are captured in files you can inspect after the job finishes. Without these flags output is often discarded.

## `srun`
Launch parallel tasks **inside** an allocation. `srun` is the primary way to start MPI jobs and to split a large allocation into smaller concurrent tasks.

Typical usage:

```bash
srun -n 4 hostname
```

In this case the command `hostname` is executed four times, once per task. One `sbatch` script can contain multiple `srun` commands executed either in serial or in parallel using the shell's background operator `&`:

```bash
#!/bin/bash
#SBATCH --output=time.out
#SBATCH --error=error.out
#SBATCH --job-name=test
#SBATCH --chdir=/u/janj/slurmtest
#SBATCH --get-user-env=L
#SBATCH --partition=s.cmfe
#SBATCH --time=60
#SBATCH --ntasks=8

srun -n 4 hostname &
srun -n 4 hostname &
wait
```

The `wait` at the end ensures the batch script does not exit (and release the allocation) before both `srun` calls have finished.

Common use cases:

* launching MPI applications
* interactive execution within an allocation obtained via `salloc`
* starting multiple tasks within one allocation to maximise node utilisation

`srun` is frequently used:

* inside batch scripts submitted with `sbatch`
* interactively after obtaining an allocation with `salloc`

A full list of options is available in the [SLURM documentation](https://slurm.schedmd.com/srun.html).

## `squeue`

Inspect the current scheduler queue.

Typical usage:

```bash
squeue -u $USER
```

Shows:

* pending jobs (state `PD`) waiting in the queue
* running jobs (state `R`) currently using nodes
* job IDs, node assignments, and time limits
* queue (partition) information

To monitor a specific job repeatedly:
```bash
watch -n 30 squeue -u $USER
```

Further details are available in the [SLURM documentation](https://slurm.schedmd.com/squeue.html).


## `mpi4py`
The [Message Passing Interface (MPI)](https://www.mpi-forum.org) is the dominant parallelisation standard on HPC systems. It allows many independent processes — potentially spread across many compute nodes — to communicate by explicitly passing data *messages*. The Python binding for MPI is provided by the [mpi4py](https://mpi4py.readthedocs.io) package. `mpi4py` exposes the MPI communicator that SLURM (via `srun`) or Flux (via `flux run`) creates when launching parallel tasks.

A typical MPI Python script is started with `srun`:
```bash
#!/bin/bash
#SBATCH --output=time.out
#SBATCH --error=error.out
#SBATCH --job-name=test
#SBATCH --chdir=/u/janj/slurmtest
#SBATCH --get-user-env=L
#SBATCH --partition=s.cmfe
#SBATCH --time=60
#SBATCH --ntasks=8

srun --mpi=pmix -n 8 python script.py
```

The `--mpi=pmix` flag tells SLURM which PMI (Process Management Interface) library to use for bootstrapping the MPI ranks. The correct value depends on how MPI was compiled on the cluster; `pmix` is common with OpenMPI 5+ and with Flux. If in doubt, check your cluster's documentation or try `--mpi=pmi2`.

Inside the Python script `script.py`:
```python
from mpi4py import MPI
comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()
print(rank, size)
```

Every MPI process runs the same Python script. Each process has a unique `rank` (integer from 0 to `size - 1`) and all processes share the same `size` (the total number of MPI tasks). The result of this script is a list of `(rank, size)` pairs printed in non-deterministic order:
```
2 8
0 8
4 8
3 8
6 8
5 8
1 8
7 8
```

> **Troubleshooting:** If the script only returns eight entries of `0 1` then the MPI backend is not correctly configured — each process is seeing its own private `MPI_COMM_WORLD` of size 1 instead of the shared communicator. This typically means the `--mpi` flag is wrong, or that the `mpi4py` installation was not compiled against the same MPI library that `srun` uses.

---

### Option 1: Split the Allocation with Multiple `srun` Calls

Run two independent four-task MPI jobs within the same eight-task allocation:

```bash
#!/bin/bash
#SBATCH --output=time.out
#SBATCH --error=error.out
#SBATCH --job-name=test
#SBATCH --chdir=/u/janj/slurmtest
#SBATCH --get-user-env=L
#SBATCH --partition=s.cmfe
#SBATCH --time=60
#SBATCH --ntasks=8

srun --mpi=pmix -n 4 python script.py &
srun --mpi=pmix -n 4 python script.py &
wait
```

Each `srun` launches a completely separate MPI job. Each script sees its own `MPI.COMM_WORLD` of size 4 and the output is:

```
2 4
3 4
1 4
0 4
2 4
1 4
0 4
3 4
```

**When to choose this:** Running separate programs or executables, or when the two tasks have no need to exchange data with each other.

**Potential pitfall:** Many `srun` launches within a single allocation can create scheduler and MPI launcher overhead. On some systems this also causes contention when many jobs do the same thing simultaneously (a common HPC workflow pattern).

### Option 2: Use One `srun` and Split Communicators in Python

Use a single `srun` to start all eight MPI processes, then divide `MPI.COMM_WORLD` into logical sub-groups inside the Python script:

```bash
#!/bin/bash
#SBATCH --output=time.out
...
#SBATCH --ntasks=8

srun --mpi=pmix -n 8 python script.py
```

Inside `script.py`:

```python
from mpi4py import MPI

def task(comm):
    rank = comm.Get_rank()
    size = comm.Get_size()
    print(rank, size)

world = MPI.COMM_WORLD
rank = world.Get_rank()
color = 0 if rank < 4 else 1
subcomm = world.Split(color=color, key=rank)
if color == 0:
    # ranks 0–3 run task A
    task(subcomm)
else:
    # ranks 4–7 run task B
    task(subcomm)
```

`MPI.COMM_WORLD.Split()` creates a new communicator that contains only the processes assigned the same `color`. Processes with `color=0` (ranks 0–3) form one sub-communicator of size 4; processes with `color=1` (ranks 4–7) form another.

**When to choose this:** All tasks are driven from one Python script and can share a single MPI launch. This approach has lower startup overhead and allows ranks to exchange data across groups using the parent `MPI.COMM_WORLD` communicator if needed.

**Key difference summary:**

| Approach | MPI launches | Can exchange data across groups? | Best for |
|---|---|---|---|
| Multiple `srun` | One per group | No (separate MPI jobs) | Different executables or programs |
| Communicator split | One total | Yes (via `COMM_WORLD`) | Same script, coordinated workflow |

> **Looking ahead:** Managing dozens or hundreds of these `srun` calls by hand quickly becomes unmanageable. The next notebooks introduce `executorlib`, which automates this scheduling on a *per-Python-function* level so you can focus on science rather than job scripts.
